In [1]:
!pip install pandas numpy scikit-learn nltk

In [2]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...


True

In [7]:
import os

os.listdir(r"C:\Users\himan\OneDrive\Desktop\sentiment analysis dataset")

['Test.csv', 'Train.csv', 'Valid.csv']

In [9]:
df = pd.read_csv(r"C:\Users\himan\OneDrive\Desktop\sentiment analysis dataset\Train.csv")
df.head()

,text,label
0,I grew up (b. 1965) watching and loving the Th...,0
1,"When I put this movie in my DVD player, and sa...",0
2,Why do people who do not know what a particula...,0
3,Even though I have great interest in Biblical ...,0
4,Im a die hard Dads Army fan and nothing will e...,1


In [10]:
df.columns

Index(['text', 'label'], dtype='object')

In [11]:
import re

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    
    text = text.lower()   # lowercase
    
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    
    text = re.sub(r"[^a-zA-Z]", " ", text)   # remove special characters
    
    words = text.split()   # tokenization
    
    words = [word for word in words if word not in stop_words]   # remove stopwords
    
    words = [lemmatizer.lemmatize(word) for word in words]   # lemmatization
    
    return " ".join(words)

In [12]:
df['clean_text'] = df['text'].apply(preprocess_text)

In [13]:
df[['text','clean_text']].head()

,text,clean_text
0,I grew up (b. 1965) watching and loving the Th...,grew b watching loving thunderbird mate school...
1,"When I put this movie in my DVD player, and sa...",put movie dvd player sat coke chip expectation...
2,Why do people who do not know what a particula...,people know particular time past like feel nee...
3,Even though I have great interest in Biblical ...,even though great interest biblical movie bore...
4,Im a die hard Dads Army fan and nothing will e...,im die hard dad army fan nothing ever change g...


NLP Preprocessing

The dataset text is cleaned using multiple NLP steps:

- Converted text to lowercase
- Removed URLs and special characters
- Tokenized the text
- Removed stopwords
- Applied lemmatization

This preprocessing helps in reducing noise and improving model performance.

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()

X_bow = bow.fit_transform(df['clean_text'])

print(X_bow.shape)

(40000, 81714)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(df['clean_text'])

print(X_tfidf.shape)

(40000, 81714)


In [19]:
y = df['label']

Feature Engineering

Text data is converted into numerical format using:

- Bag of Words (BoW): Counts word frequency
- TF-IDF: Measures importance of words

TF-IDF gives better representation by reducing importance of common words.

In [21]:
from sklearn.model_selection import train_test_split

X = X_tfidf   # using TF-IDF
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (32000, 81714)
Test size: (8000, 81714)


Train-Test Split

The dataset is split into training (80%) and testing (20%) sets.
This helps evaluate model performance on unseen data.

In [23]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [24]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [25]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))
    print("\n")

In [28]:
evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")

--- Logistic Regression ---
Accuracy : 0.892
Precision: 0.8841492971400873
Recall   : 0.9043133366385722
F1 Score : 0.8941176470588236


--- Naive Bayes ---
Accuracy : 0.866375
Precision: 0.8829759752002067
Recall   : 0.8472979672781359
F1 Score : 0.8647691334598355


--- Decision Tree ---
Accuracy : 0.720625
Precision: 0.722813970770374
Recall   : 0.7233515121467526
F1 Score : 0.7230826415561888




 Model Comparison & Insights

- Logistic Regression performed best as it handles sparse data efficiently.
- Naive Bayes is fast and works well for text data but slightly less accurate.
- Decision Tree tends to overfit and gave lower performance.

- TF-IDF performed better than Bag of Words because it reduces the impact of common words.

Overall, Logistic Regression with TF-IDF is the best model for this task.